> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업에서 쓰는 실습 노트북이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🤖 의사결정 트리 (Decision Tree)

**분류 · 지도학습** — 데이터를 가장 잘 나누는 예/아니오 질문을 골라 트리를 키운다.
좋은 질문이란 나눈 뒤의 **지니 불순도**(섞인 정도)가 가장 낮아지는 질문이에요.

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추를 누르거나 **Shift + Enter**를 누르면 그 셀이 실행돼요.
> 맨 위 셀부터 **차례로** 실행하세요. 코드를 고쳐 보고 싶다면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 사본을 만들어 두세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명을 사이트의 의사코드와 한 줄씩 맞춰 읽으면 돼요.

- 트리 모델과, 트리를 글자로 보여 주는 export_text를 불러와요.

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
df = pd.read_csv('https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins_clean.csv')

## 1. 지니 불순도 직접 계산하기

    지니 = 1 − (각 종 비율)²의 합   → 한 종뿐이면 0, 고르게 섞일수록 커져요

- 사이트 트리에서 본 뿌리, 왼쪽 노드, 젠투 잎의 지니와 같아요.

In [ ]:
def gini(counts):
    n = sum(counts)
    return 1 - sum((c / n) ** 2 for c in counts)

print('아델리5 턱끈5 젠투6 :', round(gini([5, 5, 6]), 3))
print('아델리5 턱끈5      :', round(gini([5, 5, 0]), 3))
print('젠투6만            :', round(gini([0, 0, 6]), 3))

## 2. 사이트와 같은 16마리로 트리 만들기

- criterion='gini'는 지니 불순도로 질문을 고르라는 뜻이에요.
- 사이트에서 한 단계씩 만든 트리와 똑같은 질문이 나와요.

In [ ]:
ids = [1, 6, 14, 21, 31, 277, 278, 280, 282, 285, 153, 154, 158, 160, 161, 164]
small = df[df['번호'].isin(ids)]
tree = DecisionTreeClassifier(criterion='gini', random_state=0)
tree.fit(small[['날개길이', '부리길이']], small['종'])
print(export_text(tree, feature_names=['날개길이', '부리길이']))

- 날개가 짧으니 왼쪽으로, 부리가 기니 오른쪽으로 가서 턱끈이 나와요!

In [ ]:
# 283번 펭귄(날개 178, 부리 46.1)은?
tree.predict(pd.DataFrame([[178, 46.1]], columns=['날개길이', '부리길이']))

## 3. 전체 데이터로 학습·평가

- max_depth=3은 질문을 최대 3번까지만 하라는 뜻이에요. 트리가 너무 깊으면 훈련 데이터를 외우기만 하거든요.
- 전체 데이터로 배우면 첫 질문이 "날개길이 ≤ 207.5"로 바뀌어요. 사이트의 16마리 트리(203.5)와 견줘 보세요.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

features = ['부리길이', '부리깊이', '날개길이', '몸무게']
X_train, X_test, y_train, y_test = train_test_split(df[features], df['종'], test_size=0.2, random_state=42)

model = DecisionTreeClassifier(max_depth=3, random_state=0)
model.fit(X_train, y_train)
print('정확도:', accuracy_score(y_test, model.predict(X_test)))
print(export_text(model, feature_names=features))

- 그래프의 제목이나 축 이름에 쓴 한글이 깨지지 않도록 글꼴 도구를 설치해요. 처음 한 번은 1~2분쯤 걸릴 수 있어요.

In [ ]:
!pip install -q koreanize-matplotlib
try:
    import koreanize_matplotlib  # 그래프에 한글 글꼴을 입혀 줘요
except ImportError:
    print('한글 글꼴을 불러오지 못했어요. 그래프 글자가 네모로 보일 수 있어요.')
import matplotlib.pyplot as plt

- plot_tree는 트리를 그림으로 그려 줘요. 상자 색은 그 상자에 가장 많은 종을 나타내요.
- feature_importances_는 속성마다 얼마나 중요하게 쓰였는지 보여 줘요. 날개길이가 가장 중요하게 쓰였어요.

In [ ]:
from sklearn.tree import plot_tree
plt.figure(figsize=(12, 6))
plot_tree(model, feature_names=features, class_names=list(model.classes_), filled=True, rounded=True)
plt.show()
print({f: float(v) for f, v in zip(features, model.feature_importances_.round(3))})

## 🚀 도전

- max_depth를 1, 2, 5, None으로 바꿔 정확도와 트리 모양을 견줘 보세요.
- 정답을 '성별'로 바꿔 펭귄의 성별을 맞히는 트리를 만들어 보세요.